# Logit Attribution Metrics Visualization

This notebook visualizes the logit attribution metrics data from `outputs/logit_attribution/rseed-42/metrics_logits.json`.

The data structure is:
- `system_name`: Dictionary containing layer-wise metrics
- `layer_idx`: Layer index (0-11)
- `quantity_name_mean/std`: Lists of values for each dimension of the system

We'll plot these quantities (mean ± std) as line plots with:
- X-axis: layer index
- Y-axis: metric values
- Each line represents a dimension of a system


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tsfm_lens.utils import apply_custom_style
import os
from typing import Any


In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")

In [ ]:
figs_save_dir = os.path.join("../../figs/logit_attribution/metrics")
os.makedirs(figs_save_dir, exist_ok=True)

In [ ]:
# Load the data
data_path = Path("../../outputs/logit_attribution/rseed-42/metrics_logits.json")
with open(data_path, "r") as f:
    data = json.load(f)

print("Available systems:", list(data.keys()))
print("Available layers for VallisElNino:", list(data["VallisElNino"].keys()))
print("Available metrics for layer 0:", list(data["VallisElNino"]["0"].keys()))


In [ ]:
def plot_metric_subplots(
    data: dict[str, dict[str, dict[str, Any]]],
    metric_name: str,
    figsize: tuple[int, int] = (5, 5),
    title: str | None = None,
    save_path: str | None = None,
    colors: list[str] | None = None,
    show_std_envelope: bool = True,
    line_alpha: float = 0.5,
    show_markers: bool = True,
    show_ylabel: bool = True,
) -> None:
    """
    Plot a single metric across layers, systems, and dimensions.
    """
    fig, ax = plt.subplots(figsize=figsize)

    metric_name_title = metric_name.replace("_", " ").title()
    if metric_name == "entropy":
        metric_name_title = "Avg Entropy"
    elif metric_name == "top_k_entropy":
        metric_name_title = "Avg Top-10 Entropy"
    elif metric_name == "effective_vocab":
        metric_name_title = r"Effective Vocab ($n_\mathrm{tokens}$ : $p > 0.01$)"

    # Collect all unique layer indices
    layers = sorted({int(l) for sys in data.values() for l in sys.keys()})

    if colors is None:
        colors = ["black"]

    # Store all means for calculating median
    all_means = []

    num_samples_plotted = 0
    for i, (system_name, system_data) in enumerate(data.items()):
        first_layer = str(layers[0])
        if first_layer not in system_data:
            continue
        n_dims = len(system_data[first_layer][f"{metric_name}_mean"])
        for j, dim_idx in enumerate(range(n_dims)):
            num_samples_plotted += 1
            means, stds = [], []
            for layer in layers:
                layer_str = str(layer)
                if layer_str in system_data:
                    means.append(system_data[layer_str][f"{metric_name}_mean"][dim_idx])
                    stds.append(system_data[layer_str][f"{metric_name}_std"][dim_idx])
                else:
                    means.append(np.nan)
                    stds.append(np.nan)
            all_means.append(means)
            ax.plot(
                layers,
                means,
                label=f"{system_name} - Dim {dim_idx}",
                marker="o" if show_markers else None,
                linewidth=1,
                markersize=2,
                alpha=line_alpha,
                color=colors[num_samples_plotted % len(colors)],
            )
            ax.fill_between(
                layers,
                np.array(means) - np.array(stds),
                np.array(means) + np.array(stds),
                alpha=0.2 * line_alpha,
                color=colors[num_samples_plotted % len(colors)],
            )

    # Plot median line
    median_values = np.nanmedian(np.array(all_means), axis=0)
    ax.plot(layers, median_values, color="#ff3333", linewidth=3, zorder=100, label="Median", marker="o", markersize=6)

    print(f"num_samples_plotted: {num_samples_plotted}")
    ax.set_xlabel("Layer Index", fontweight="bold")
    ax.set_xticks(layers)
    ax.set_ylabel(metric_name_title, fontweight="bold") if show_ylabel else None
    ax.set_title(
        title or f"{metric_name_title} Across Layers",
        fontsize=12,
        fontweight="bold",
    )
    # ax.grid(True, alpha=0.3)
    # ax.legend(fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    plt.show()

In [ ]:
# plot_metric_subplots(
#     data,
#     "entropy",
#     title="DLA (Residual Stream)",
#     line_alpha=0.8,
#     show_markers=False,
#     colors=DEFAULT_COLORS,
# )

plot_metric_subplots(
    data,
    "entropy",
    # title="DLA (Residual Stream)",
    line_alpha=0.03,
    show_markers=False,
    show_ylabel=False,
    save_path=os.path.join(figs_save_dir, "entropy.pdf"),
    figsize=(4, 6),
)


In [ ]:
plot_metric_subplots(
    data,
    "top_k_entropy",
    title="DLA (Residual Stream)",
    line_alpha=0.05,
    show_markers=False,
    save_path=os.path.join(figs_save_dir, "top_k_entropy.pdf"),
)

plot_metric_subplots(
    data,
    "effective_vocab",
    title="DLA (Residual Stream)",
    line_alpha=0.05,
    show_markers=False,
    save_path=os.path.join(figs_save_dir, "effective_vocab.pdf"),
)


In [ ]:
plot_metric_subplots(
    data,
    "peak_counts",
    # title="DLA (Residual Stream)",
    line_alpha=0.03,
    show_markers=False,
    show_ylabel=False,
    save_path=os.path.join(figs_save_dir, "peak_counts.pdf"),
    figsize=(4, 6),
)

# Attribution Metrics Visualization

This section visualizes the attribution metrics data from `outputs/logit_attribution/rseed-42_debug/metrics_attribution.json`.

The data structure is:
- `read_stream`: Dictionary containing layer-wise attribution metrics
- `layer_idx`: Layer index (0-11)
- `prediction_horizon`: Prediction length in multiples of 64 (64, 128, 192, 256, 320, 384, 448, 512)
- `system_name`: System name (VallisElNino, Blasius, LiuChen, Laser)
- `metric_name`: Metric name (mse, rmse, mae, smape, spearman)

We'll plot these quantities with:
- X-axis: prediction horizon
- Y-axis: metric values
- Each line represents a system
- Separate plots for each metric and layer


In [ ]:
# Load the attribution metrics data
attribution_data_path = Path("../../outputs/logit_attribution/rseed-42/metrics_attribution.json")
with open(attribution_data_path, "r") as f:
    attribution_data = json.load(f)

print("Available streams:", list(attribution_data.keys()))
print("Available layers for read_stream:", list(attribution_data["read_stream"].keys()))
print(
    "Available prediction horizons for layer 0:",
    list(attribution_data["read_stream"]["0"].keys()),
)
print(
    "Available systems for layer 0, horizon 64:",
    list(attribution_data["read_stream"]["0"]["64"].keys()),
)
print(
    "Available metrics for VallisElNino:",
    list(attribution_data["read_stream"]["0"]["64"]["VallisElNino"].keys()),
)


In [ ]:
def plot_attribution_metric_across_layers_at_horizon(
    attribution_data: dict,
    metric_name: str,
    prediction_horizon: int = 512,
    figsize: tuple[int, int] = (5, 5),
    title: str | None = None,
    colors: list[str] | None = None,
    show_markers: bool = True,
    line_alpha: float = 0.5,
    outlier_thresh: float = 3.0,  # Z-score threshold for outlier exclusion
) -> None:
    """Plot percent gain of a metric across layers at a fixed prediction horizon (excluding layer 0), excluding outliers."""
    fig, ax = plt.subplots(figsize=figsize)
    layers = sorted(map(int, attribution_data["read_stream"].keys()))
    systems = list(attribution_data["read_stream"][str(layers[0])][str(prediction_horizon)].keys())
    if colors is None:
        colors = ["black"]

    # Collect all percent_gain values for all systems for outlier detection
    all_percent_gains = []
    percent_gain_by_system = {}

    for i, system in enumerate(systems):
        raw = [
            attribution_data["read_stream"][str(layer)]
            .get(str(prediction_horizon), {})
            .get(system, {})
            .get(metric_name, np.nan)
            for layer in layers
        ]
        # Compute percent gain, skip layer 0
        percent_gain = [
            ((raw[j] - raw[j - 1]) / raw[j - 1]) * 100
            if j > 0 and not (np.isnan(raw[j]) or np.isnan(raw[j - 1]) or raw[j - 1] == 0)
            else np.nan
            for j in range(1, len(raw))
        ]
        percent_gain_by_system[system] = percent_gain
        all_percent_gains.extend([v for v in percent_gain if not np.isnan(v)])

    # Outlier exclusion: compute z-scores and mask outliers
    if all_percent_gains:
        all_percent_gains_arr = np.array(all_percent_gains)
        mean = np.nanmean(all_percent_gains_arr)
        std = np.nanstd(all_percent_gains_arr)

        def is_outlier(val):
            if np.isnan(val):
                return False
            if std == 0:
                return False
            return abs((val - mean) / std) > outlier_thresh
    else:

        def is_outlier(val):
            return False

    for i, system in enumerate(systems):
        percent_gain = percent_gain_by_system[system]
        # Mask outliers with np.nan
        percent_gain_no_outliers = [v if not is_outlier(v) else np.nan for v in percent_gain]
        ax.plot(
            layers[1:],
            percent_gain_no_outliers,
            label=system,
            marker="o" if show_markers else None,
            linewidth=1,
            markersize=2,
            alpha=line_alpha,
            color=colors[i % len(colors)],
        )
    metric_title = metric_name.upper() if metric_name in {"mse", "rmse", "mae"} else metric_name.title()
    if metric_name == "smape":
        metric_title = "sMAPE"
    ax.set_xlabel("Layer Index", fontweight="bold")
    plt.xticks(layers[1:])
    ax.set_ylabel(f"{metric_title} % Gain", fontweight="bold")
    ax.axhline(y=0, color="black", linestyle="--", alpha=0.5)
    ax.set_title(
        title
        or rf"$\%\ \Delta \mathrm{{{metric_title}}}\ \text{{Gain over Prev Layer}}\ (L_{{\mathrm{{pred}}}}={prediction_horizon})$",
        fontsize=13,
        fontweight="bold",
    )
    plt.tight_layout()


In [ ]:
# Plot individual metrics across layers at horizon 512
plot_attribution_metric_across_layers_at_horizon(
    attribution_data,
    "smape",
    512,
    show_markers=False,
    line_alpha=0.1,
    outlier_thresh=5.0,
)


In [ ]:
plot_attribution_metric_across_layers_at_horizon(
    attribution_data, "mae", 512, show_markers=False, line_alpha=0.1, outlier_thresh=5.0
)

In [ ]:
plot_attribution_metric_across_layers_at_horizon(
    attribution_data,
    "spearman",
    512,
    show_markers=False,
    line_alpha=0.1,
    outlier_thresh=5.0,
)


In [ ]:
plot_attribution_metric_across_layers_at_horizon(
    attribution_data,
    "rmse",
    512,
    show_markers=False,
    line_alpha=0.1,
    outlier_thresh=5.0,
)
